# Compare V1 and V2 Implementations

In [ ]:
# Benchmarking: Compare runtime and output consistency for different node sizes
import numpy as np
import pandas as pd
import time
from fastdag2pag_V2.dag2 import dag2
from fastdag2pag_V1.dag2pag import dag2pag
from fastdag2pag_V1.Random_Graph import ErdosRenyi
import matplotlib.pyplot as plt

def binarize_adj(adj: pd.DataFrame) -> pd.DataFrame:
    """Convert all nonzero elements of the adjacency matrix to 1, keep 0 unchanged."""
    return (adj != 0).astype(int)


def check_PMG(pmg) -> bool:
    """
    Verify consistency between _adj and _pattern_index.
    Returns True if consistent, else False.
    """
    # Check if each edge in _adj has a corresponding entry in _pattern_index
    for u in pmg._adj:
        for v, marks in pmg._adj[u].items():
            mark_u, mark_v = marks
            # Check u's index
            if v not in pmg._pattern_index[u][marks]:
                return False
            # Check v's index (symmetric)
            if u not in pmg._pattern_index[v][(mark_v, mark_u)]:
                return False
    
    # Check if each entry in _pattern_index has a corresponding edge in _adj
    for u in pmg._pattern_index:
        for pattern, neighbors in pmg._pattern_index[u].items():
            for v in neighbors:
                # Check if corresponding edge exists in _adj
                if v not in pmg._adj[u] or pmg._adj[u][v] != pattern:
                    return False
                # Check symmetric edge
                mark_u, mark_v = pattern
                if u not in pmg._adj[v] or pmg._adj[v][u] != (mark_v, mark_u):
                    return False
    
    return True




node_sizes = [30, 40, 50, 100, 150]
num_trials = 100
results = []



for n_nodes in node_sizes:
    dag2pag_times = []
    dag_to_pag_times = []
    pmg_record = []
    match_count = 0
    ER_graph_gen = ErdosRenyi(n_nodes, expected_degree=2, def_dataframe=True, seed=321)
    for trial in range(num_trials):
        # Generate random DAG adjacency matrix
        graph = ER_graph_gen.get_random_graph()
        num_latent = int(n_nodes * 0.1) if n_nodes >= 10 else 1
        num_sel = num_latent  # For simplicity, set number of selection bias nodes equal to number of latent nodes
        adj = ER_graph_gen.set_latent_nodes(graph, num_latent=num_latent, selection_bias=True, num_sel=num_sel)
        latent_nodes = [node for node in adj.columns if node.startswith('L')]
        selection_bias_nodes = [node for node in adj.columns if node.startswith('S')]

        # DAG_to_PAG (causal-learn)
        t0 = time.time()
        learner = dag2(adj, latent_nodes=latent_nodes, selection_bias_nodes=selection_bias_nodes)
        res = learner.dag2pag()
        t1 = time.time()
        pag1 = res['PAG.DataFrame']
        check_pmg = check_PMG(res['PAG'])
        pmg_record.append(check_pmg)

        
        dag_to_pag_times.append(t1 - t0)

        # FastDAG2PAG
        t2 = time.time()
        pag2 = dag2pag(adj, latent_nodes=latent_nodes, selection_bias=selection_bias_nodes)['PAG.DataFrame']
        t3 = time.time()
        dag2pag_times.append(t3 - t2)

        # Compare outputs (matrix equality, ignoring edge marks)
        if binarize_adj(pag1).equals(binarize_adj(pag2)):
            match_count += 1
        else:
            print(f"Discrepancy found in graph {n_nodes}-{trial+1}")
            diff = pag1.compare(pag2, result_names=("version2.0", "version1.0"))
            print(diff)

    results.append({
        'nodes': n_nodes,
        'fastdag2pag_V2_avg_time': np.mean(dag_to_pag_times),
        'fastdag2pag_V1_avg_time': np.mean(dag2pag_times),
        'match_ratio': match_count / num_trials,
        'pmg.consistency': pmg_record.count(True)/ num_trials  # True if all PMG checks passed
    })

results_df = pd.DataFrame(results)
print('Match ratio (PAG matrix equality) for each node size:')
for index, row in results_df.iterrows():
    print(f"Node Size: {row['nodes']}, Match Ratio: {row['match_ratio']:.2f}")
    print(f"PMG consistency check passed: {row['pmg.consistency']:.2f}")

In [ ]:
# Plot the average runtime comparison between the two methods
import matplotlib as mpl
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.edgecolor'] = 'black'
mpl.rcParams['axes.labelsize'] = 13
mpl.rcParams['axes.titlesize'] = 15
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.facecolor'] = 'white'

# Nature-style color palette
nature_colors = ['#3C5488', '#E64B35']
ax = results_df.plot(x='nodes', y=['fastdag2pag_V2_avg_time', 'fastdag2pag_V1_avg_time'], kind='bar',
                    color=nature_colors,
                    edgecolor='none',
                    figsize=(7,4),
                    legend=True)
ax.set_ylabel('Average Time (s)', fontsize=13, fontname='Arial')
ax.set_xlabel('Number of Nodes', fontsize=13, fontname='Arial')
ax.set_title('Average Runtime Comparison', fontsize=15, fontweight='bold', fontname='Arial')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(['V2', 'V1'], fontsize=12, frameon=False)
ax.set_xticklabels(results_df['nodes'], rotation=90, fontsize=12, fontname='Arial')
ax.tick_params(axis='y', labelsize=12, width=1.5)
ax.tick_params(axis='x', width=1.5)
# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()